# Spatial Statistics Final Project — GPU REML Notebook
This notebook is aligned with the project methodology in the PDF: model each 50×50 subregion with IRF-2, use the sign-flipping generalized covariance, use the basis M=[1,x,y,x^2,y^2,xy], optimize alpha in (0.01,5.99), and report kappa = alpha/2.

In [3]:
!pip -q install rasterio plotly kaleido numpy matplotlib

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import math, time
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
import torch
import plotly.express as px

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ModuleNotFoundError: No module named 'numpy'

In [ ]:
tif_file = 'USGS_13_n37w118_20260112.tif'
out_csv = 'hw9_gpu_project_results.csv'
block_size = 50
thin_step = 2
alpha_lower, alpha_upper = 0.01, 5.99
alpha_steps = 120
max_blocks = None  # set to 100 for a quick test

## Core functions
These match the PDF methodology: precompute the distance matrix and the IRF-2 basis once because they are identical for all 1600 subregions.[file:267]

In [ ]:
def create_M(grid_size=25, device=None, dtype=torch.float64):
    xs, ys = torch.meshgrid(
        torch.arange(1, grid_size + 1, device=device, dtype=dtype),
        torch.arange(1, grid_size + 1, device=device, dtype=dtype),
        indexing="xy"
    )
    x = xs.reshape(-1)
    y = ys.reshape(-1)
    return torch.stack([torch.ones_like(x), x, y, x*x, y*y, x*y], dim=1)

def create_D(grid_size=25, device=None, dtype=torch.float64):
    xs, ys = torch.meshgrid(
        torch.arange(1, grid_size + 1, device=device, dtype=dtype),
        torch.arange(1, grid_size + 1, device=device, dtype=dtype),
        indexing="xy"
    )
    coords = torch.stack([xs.reshape(-1), ys.reshape(-1)], dim=1)
    diff = coords[:, None, :] - coords[None, :, :]
    return torch.sqrt((diff * diff).sum(dim=2))

def omega_from_alpha(alpha, D):
    av = float(alpha.item())
    if 0 < av < 2:
        Omega = -(D ** av)
    elif 2 <= av < 4:
        Omega = +(D ** av)
    else:
        Omega = -(D ** av)
    Omega = Omega.clone()
    idx = torch.arange(Omega.shape[0], device=Omega.device)
    Omega[idx, idx] += 1e-6
    return Omega

def restricted_loglik_alpha(alpha, z_vec, D, M):
    n = z_vec.numel()
    p = M.shape[1]
    Omega = omega_from_alpha(alpha, D)
    try:
        L = torch.linalg.cholesky(Omega)
        Omega_inv = torch.cholesky_inverse(L)
    except RuntimeError:
        return None

    MO = M.transpose(0, 1) @ Omega_inv
    MOM = MO @ M
    try:
        Lm = torch.linalg.cholesky(MOM)
        MOM_inv = torch.cholesky_inverse(Lm)
    except RuntimeError:
        return None

    beta_hat = MOM_inv @ MO @ z_vec
    residuals = z_vec - (M @ beta_hat)
    qf = (residuals.transpose(0, 1) @ Omega_inv @ residuals).squeeze()
    theta = qf / (n - p)
    if (not torch.isfinite(theta)) or theta <= 0:
        return None

    sign_o, logdet_o = torch.linalg.slogdet(Omega)
    sign_m, logdet_m = torch.linalg.slogdet(MOM)
    if sign_o <= 0 or sign_m <= 0:
        return None

    return 0.5 * logdet_o + 0.5 * logdet_m + ((n - p) / 2.0) * torch.log(qf)

def optimize_alpha_grid(z_vec_np, D, M, lower=0.01, upper=5.99, steps=120):
    z_vec = torch.tensor(z_vec_np.reshape(-1, 1), device=D.device, dtype=torch.float64)
    grid = torch.linspace(lower, upper, steps=steps, device=D.device, dtype=torch.float64)
    best_val = None
    best_alpha = None
    for alpha in grid:
        val = restricted_loglik_alpha(alpha, z_vec, D, M)
        if val is None:
            continue
        if (best_val is None) or (val < best_val):
            best_val = val
            best_alpha = alpha
    if best_val is None:
        return np.nan, np.nan, np.nan, False
    alpha_hat = float(best_alpha.item())
    kappa_hat = alpha_hat / 2.0
    return round(alpha_hat, 2), round(kappa_hat, 2), float(best_val.item()), True

## Read the 2000×2000 raster and build the 1600 blocks
Since the new data are already 2000×2000, this notebook uses the raster directly and does not aggregate or crop first. That is the main preprocessing difference from the prior code.[file:267]

In [ ]:
with rasterio.open(tif_file) as src:
    Z = src.read(1)

print("Raster shape:", Z.shape)
if Z.shape != (2000, 2000):
    raise ValueError(f"Expected a 2000x2000 raster, got {Z.shape}")

n_row_blocks = Z.shape[0] // block_size
n_col_blocks = Z.shape[1] // block_size
print("Block grid:", n_row_blocks, "x", n_col_blocks, "=", n_row_blocks * n_col_blocks)

blocks = []
meta = []
for r in range(n_row_blocks):
    for c in range(n_col_blocks):
        rs, re = r * block_size, (r + 1) * block_size
        cs, ce = c * block_size, (c + 1) * block_size
        sub = Z[rs:re, cs:ce]
        sub = sub[::thin_step, ::thin_step]
        blocks.append(sub.reshape(-1).astype(np.float64))
        meta.append((r + 1, c + 1))

if max_blocks is not None:
    blocks = blocks[:max_blocks]
    meta = meta[:max_blocks]
    print("Using subset of blocks:", len(blocks))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
D = create_D(grid_size=25, device=device)
M = create_M(grid_size=25, device=device)

rows = []
t0 = time.time()
for i, ((rb, cb), z) in enumerate(zip(meta, blocks), start=1):
    alpha_hat, kappa_hat, reml_obj, converged = optimize_alpha_grid(
        z, D, M, lower=alpha_lower, upper=alpha_upper, steps=alpha_steps
    )
    rows.append({
        "row_block": rb,
        "col_block": cb,
        "alpha_hat": alpha_hat,
        "kappa_hat": kappa_hat,
        "reml_objective": reml_obj,
        "converged": converged
    })
    if i % 50 == 0:
        print(f"Finished {i}/{len(blocks)} blocks in {time.time() - t0:.1f} sec")

results = pd.DataFrame(rows)
results.to_csv(out_csv, index=False)
print(results.head())
print(results[["alpha_hat", "kappa_hat", "reml_objective"]].describe(include="all"))

In [ ]:
heat = results.pivot(index="row_block", columns="col_block", values="kappa_hat").sort_index()
heat = heat.iloc[::-1]
fig1 = px.imshow(heat, aspect="auto", origin="lower", color_continuous_scale="Viridis", labels={"x":"Column block","y":"Row block","color":"Kappa"}, title="Spatial map of kappa estimates")
fig1.show()

fig2 = px.histogram(results, x="kappa_hat", nbins=30, title="Histogram of kappa estimates")
fig2.show()

In [ ]:
from google.colab import files
files.download(out_csv)